# Guillotine week projection

Projects **points for every rostered player** in the target week, plus each live team's chance of being **eliminated** and of finishing **first** (highest score of the week). It also lets you test **what-if lineup moves** for your own team.

**How it works**
1. Trains the PPG model on all seasons 2021-2025 (validation on 2024 picks the tree count, then a refit on everything).
2. Builds the week's features with the same pipeline used for training: season-to-date points, snap %, carries, targets and return stats per game (weeks before the target week only), plus FFC superflex ADP.
3. Team score = sum of the 10 starters' projected points. Each player carries an independent error of about the model's out-of-sample RMSE, so the team score is simulated 20,000 times and every team is ranked in each simulation.

**League-scored points.** Every player's raw Sleeper stats are scored with *your league's own* scoring settings (interceptions -2, kicker and defense rules, return yards). This reproduces Sleeper's league points exactly, so the model's target, its season-to-date features and free-agent pickups all use the same scoring as the live scores.

**Live view.** The export cell near the end writes `live_inputs_<year>_wk<week>.json`. `live_app.py` (Streamlit) combines it with Sleeper's live scores and the NFL game clocks to show live elimination / last-place / first-place odds. Re-run this notebook before each week's games.

**Settings (first code cell)**
- `ELIMINATIONS_BY_WEEK` / `N_ELIMINATED`: how many teams are cut (1 per week, 2 in weeks 2, 4, 5, 9, 12, 14).
- `IMMUNE_TEAMS`: usernames that cannot be cut this week. They still play and can still finish first.
- `MY_TEAM` + `SCENARIOS`: lineup swaps, waiver pickups and trades to test. Every scenario is compared with your current lineup using the *same* random draws, so the differences are due to the lineup change and not simulation noise.

**Assumptions to know about**
- Players on a bye or designated Out / IR / PUP / Sus / NA are projected 0. Questionable and Doubtful players are *not* adjusted (see `injury_status`).
- Independent errors ignore correlation between teammates (e.g. a QB and his receivers), which makes the probabilities somewhat too confident.
- Free agents you swap in use the same features and model as rostered players.
- **Trades:** the other team automatically plays its best legal lineup afterwards (this can also change their score for reasons unrelated to the players moved, if they were not already starting their best lineup). Your own lineup keeps its other starters and fills any hole with your best legal option; use `swaps` in the scenario to start someone else, including a player you just acquired. Roster-size limits and league trade rules are not checked.


In [ ]:
# ===== SETTINGS =====
TARGET_YEAR = 2026
TARGET_WEEK = None            # None = the current NFL week according to Sleeper
N_SIMS = 20000

# Teams cut per week: 1, except 2 in weeks 2, 4, 5, 9, 12 and 14
ELIMINATIONS_BY_WEEK = {w: (2 if w in (2, 4, 5, 9, 12, 14) else 1) for w in range(1, 19)}
N_ELIMINATED = None           # None = use ELIMINATIONS_BY_WEEK; or a number to override this week

# Usernames (not case-sensitive) of teams that can NOT be eliminated this week.
# Immune teams still play and can still finish first; they are just skipped when the cut is chosen.
IMMUNE_TEAMS = []             # e.g. ["twilke18", "Boots1991"]

# What-if moves for MY_TEAM. A scenario is either
#   * a list of lineup swaps:  [(player out, player in), ...]
#   * or a dict with any of:
#         "trade":  {"with": "username", "give": [players leaving your roster], "get": [players leaving theirs]}
#         "trades": [{...}, {...}]                     # SEVERAL trades in one scenario, applied in order
#         "swaps":  [(player out, player in), ...]     # your lineup moves, applied AFTER the trades
#   (A dict can't hold the same key twice - the last one silently wins - so use "trades": [...] for more than one.)
# Swap rules: "player out" must be a current starter; "player in" can be on your bench, a free agent / waiver
# pickup, or a player you acquire in the same scenario's trade. Players can be given as a name or a Sleeper id.
# Trades: the other team automatically plays its best legal lineup afterwards. If you trade away a starter,
# the hole is filled with your best legal option (your other starters stay put); use "swaps" to start someone else.
# Lineups must stay legal (2 RB, 2 WR, TE, FLEX, 2 SUPER_FLEX, K, DEF). Roster-size limits are not checked.
MY_TEAM = "vikingsfan14"
SCENARIOS = {
    "Bourne in for Turpin":       [("KaVontae Turpin", "Kendrick Bourne")],
    "Goedert in for Waller":      [("Darren Waller", "Dallas Goedert")],
    "Williams + Bourne in for Sutton + Turpin":
                                  [("Courtland Sutton", "Kyle Williams"), ("KaVontae Turpin", "Kendrick Bourne")],
    "Trade Sutton + Bourne for Smith-Njigba": {
        "trade": {"with": "michaelwodka", "give": ["Courtland Sutton", "Kendrick Bourne"], "get": ["Jaxon Smith-Njigba"]},
    },
    "Trade Herbert for Allen": {
        "trade": {"with": "BeerMeToo", "give": ["Justin Herbert"], "get": ["Josh Allen"]},
    },
    "Both trades": {
        "trades": [
            {"with": "michaelwodka", "give": ["Courtland Sutton", "Kendrick Bourne"], "get": ["Jaxon Smith-Njigba"]},
            {"with": "BeerMeToo", "give": ["Justin Herbert"], "get": ["Josh Allen"]},
        ],
    },
    # waiver / free agent pickup:  "Pick up X for Turpin": [("KaVontae Turpin", "Some Freeagent")],
}


In [ ]:
# points_prediction_xgb.py
# Requirements:
# pip install pandas numpy scikit-learn xgboost matplotlib requests beautifulsoup4 lxml

import requests
import pandas as pd
import numpy as np
import re
import os
import json
import time
from collections import defaultdict
from sklearn.metrics import mean_squared_error
import xgboost as xgb
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

# ====== USER CONFIG ======
USERNAME = "738005499156590592"
YEARS = [2021, 2022, 2023, 2024, 2025]   # seasons to include; last one will be used as holdout by default
LEAGUE_NAME_FILTER = "Guillotine League"
MAX_WEEKS = 18
FFC_TEAMS = 14
# FFC format used for ADP. "2qb" (superflex) matches this league (2 SUPER_FLEX slots) and is the only
# format with full lists for every season 2021-2026; "ppr" ADP predicted equally well in backtests.
FFC_SCORING = "2qb"

# ====== HELPERS (from your code) ======
def normalize_name_for_match(name: str):
    if not name:
        return ""
    s = name.lower().strip()
    s = re.sub(r'\b(jr|sr|ii|iii|iv|v)\b\.?', '', s)
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def load_sleeper_players():
    url = "https://api.sleeper.app/v1/players/nfl"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json() or {}
    norm_to_id = {}
    id_to_meta = {}
    for pid, info in data.items():
        id_to_meta[pid] = info
        first = (info.get("first_name") or "").strip()
        last = (info.get("last_name") or "").strip()
        candidates = []
        if first and last:
            candidates.append(f"{first} {last}")
            candidates.append(f"{last} {first}")
        full = (info.get("full_name") or "").strip()
        if full:
            candidates.append(full)
        display = f"{first} {last}".strip()
        if display:
            candidates.append(display)
        if info.get("search_name"):
            candidates.append(info.get("search_name"))
        for c in candidates:
            n = normalize_name_for_match(c)
            if n and n not in norm_to_id:
                norm_to_id[n] = pid
    print(f"Loaded {len(norm_to_id)} normalized player names from Sleeper.")
    return norm_to_id, id_to_meta

SLEEPER_NAME_MAP, PLAYERS_META = load_sleeper_players()

def fetch_ffcalculator_adp(year, teams=FFC_TEAMS, scoring_format=FFC_SCORING):
    """
    FantasyFootballCalculator ADP as {normalized name: adp}. Every good response is saved to
    adp_cache/ so an API outage (or a thin in-season window) falls back to the last good list.
    """
    url = f"https://fantasyfootballcalculator.com/api/v1/adp/{scoring_format}?teams={teams}&year={year}"
    cache_path = os.path.join("adp_cache", f"ffc_{scoring_format}_{teams}_{year}.json")
    players = []
    try:
        data = requests.get(url, timeout=20).json()
        players = data.get("players", []) if isinstance(data, dict) else []
    except Exception:
        players = []
    if len(players) >= 50:
        try:
            os.makedirs("adp_cache", exist_ok=True)
            with open(cache_path, "w", encoding="utf-8") as fh:
                json.dump(players, fh)
        except Exception:
            pass
    elif os.path.exists(cache_path):
        print(f"[FFC] {year}: live API returned {len(players)} players; using cached list {cache_path}")
        with open(cache_path, encoding="utf-8") as fh:
            players = json.load(fh)
    adp_map = {}
    for rec in players:
        name = rec.get("name") or rec.get("player_name") or ""
        adp_val = rec.get("adp") or rec.get("average") or rec.get("adp_avg")
        if not name or adp_val is None:
            continue
        try:
            adp_map[normalize_name_for_match(name)] = float(adp_val)
        except Exception:
            pass
    return adp_map

def fetch_fantasypros_adp(year, scoring="ppr", position_scope="overall"):
    """
    Scrape FantasyPros ADP page for given scoring & overall scope and return normalized-name -> adp.
    NOTE: FantasyPros pages usually reflect the current season; if you're running for 2025,
    the page should already show 2025 ADP. This function parses the table defensively.
    """
    # build URL (we use the PPR overall page if scoring == ppr)
    if scoring.lower() == "ppr":
        url = f"https://www.fantasypros.com/nfl/adp/ppr-{position_scope}.php"
    else:
        url = f"https://www.fantasypros.com/nfl/adp/{position_scope}.php"

    try:
        resp = requests.get(url, timeout=20)
        if resp.status_code != 200:
            print(f"[FP] Error fetching FantasyPros ADP page: {resp.status_code}")
            return {}
        soup = BeautifulSoup(resp.text, "lxml")
    except Exception as e:
        print(f"[FP] Exception fetching ADP page: {e}")
        return {}

    adp_map = {}

    # Attempt 1: find table with id 'data' (common pattern)
    table = soup.find("table", {"id": "data"})
    rows = []
    if table:
        tbody = table.find("tbody") or table
        rows = tbody.find_all("tr")
    else:
        # Fallback: find first large table
        all_tables = soup.find_all("table")
        if all_tables:
            # pick the one with most rows
            best = max(all_tables, key=lambda t: len(t.find_all("tr")))
            tbody = best.find("tbody") or best
            rows = tbody.find_all("tr")

    # detect columns by header names
    headers = []
    if table:
        header_row = table.find("thead")
        if header_row:
            headers = [th.get_text(strip=True).lower() for th in header_row.find_all("th")]
    
    avg_idx = None
    for i, h in enumerate(headers):
        if "avg" in h or "adp" in h:
            avg_idx = i
            break
    
    for row in rows:
        cols = row.find_all("td")
        if not cols or len(cols) < 3:
            continue
    
        # extract name
        name = None
        for c in cols[:3]:
            a = c.find("a")
            if a and a.get_text(strip=True):
                name = a.get_text(strip=True)
                break
        if not name:
            name = cols[1].get_text(strip=True) if len(cols) > 1 else cols[0].get_text(strip=True)
    
        # pick the correct ADP column
        adp_text = None
        if avg_idx is not None and avg_idx < len(cols):
            adp_text = cols[avg_idx].get_text(strip=True)
        else:
            # fallback to last column if header not found
            adp_text = cols[-1].get_text(strip=True)
    
        m = re.search(r'(\d+(\.\d+)?)', adp_text or "")
        if not m:
            continue
    
        try:
            adp_val = float(m.group(1))
        except ValueError:
            continue
    
        n = normalize_name_for_match(name)
        if n:
            adp_map[n] = adp_val

    print(f"[FP] Scraped {len(adp_map)} ADPs from FantasyPros (scoring={scoring}, scope={position_scope})")
    return adp_map

# ====== Build per-week player stats (points + usage) ======
POINT_KEYS = ("pts_ppr", "fantasy_points", "pts", "ppr", "points")
# League scoring adds 1 pt per 10 return yards; Sleeper's pts_ppr does not include return yardage
RETURN_PTS_PER_YD = 0.1
USAGE_STAT_KEYS = ("gp", "rush_att", "rec_tgt", "off_snp", "tm_off_snp", "kr", "kr_yd", "pr", "pr_yd")

# Season-to-date usage features (all use weeks BEFORE the row's week; NaN if no games played yet)
USAGE_FEATURES = [
    "rush_att_per_game_to_date",
    "tgt_per_game_to_date",
    "snap_pct_to_date",
    "kr_per_game_to_date",
    "kr_yd_per_game_to_date",
    "pr_per_game_to_date",
    "pr_yd_per_game_to_date",
]

# Features forced to be monotonically INCREASING in the model
MONOTONE_INCREASING = ["pts_per_game_to_date"] + USAGE_FEATURES

_WEEKLY_STATS_CACHE = {}
_SCORING_CACHE = {}

def get_league_scoring(year):
    """scoring_settings of this user's league for `year` (None if the league can't be found)."""
    if year not in _SCORING_CACHE:
        scoring = None
        try:
            leagues = requests.get(f"https://api.sleeper.app/v1/user/{USERNAME}/leagues/nfl/{year}", timeout=20).json() or []
            L = next((l for l in leagues if LEAGUE_NAME_FILTER.lower() in (l.get("name") or "").lower()), leagues[0] if leagues else None)
            scoring = (L or {}).get("scoring_settings")
        except Exception:
            scoring = None
        if not scoring:
            print(f"[scoring] WARNING: no league scoring found for {year}; falling back to default PPR + return yards")
        _SCORING_CACHE[year] = scoring
    return _SCORING_CACHE[year]

def league_points(stats, scoring):
    """Fantasy points for one player-week: sum of stat * weight. Reproduces Sleeper's own league points exactly."""
    return sum(float(stats[k]) * w for k, w in scoring.items() if w and stats.get(k) is not None)

def _num(v):
    try:
        return float(v) if v is not None else 0.0
    except Exception:
        return 0.0

def fetch_weekly_stats(year, max_weeks=MAX_WEEKS):
    """
    Returns: dict week -> dict player_id -> {"pts": float, <USAGE_STAT_KEYS>: float}
    Fetched once per (year, max_weeks) and cached, since both the points and usage builders need it.
    NOTE: Sleeper omits pts_ppr for players who scored 0, so a missing value means 0 points.
    """
    key = (year, max_weeks)
    if key in _WEEKLY_STATS_CACHE:
        return _WEEKLY_STATS_CACHE[key]
    out = {}
    scoring = get_league_scoring(year)
    for week in range(1, max_weeks + 1):
        url = f"https://api.sleeper.app/v1/stats/nfl/regular/{year}/{week}"
        try:
            r = requests.get(url, timeout=20).json() or {}
        except Exception:
            r = {}
        week_map = {}
        if isinstance(r, dict):
            for pid, p in r.items():
                rec = {"pts": league_points(p, scoring) if scoring else _num(next((p[k] for k in POINT_KEYS if p.get(k)), 0))}
                for k in USAGE_STAT_KEYS:
                    rec[k] = _num(p.get(k))
                if not scoring:            # fallback only: default PPR + return yards
                    rec["pts"] += RETURN_PTS_PER_YD * (rec["kr_yd"] + rec["pr_yd"])
                # Snap counts are only present for some players; keep "unknown team snaps" distinct from 0
                rec["has_team_snaps"] = bool(p.get("tm_off_snp"))
                week_map[str(pid)] = rec
        out[week] = week_map
        time.sleep(0.1)
    _WEEKLY_STATS_CACHE[key] = out
    return out

def build_weekly_player_points(year, max_weeks=MAX_WEEKS):
    """
    Returns: dict week -> dict player_id -> points (float) for that week (ppr preferred)
    """
    stats = fetch_weekly_stats(year, max_weeks)
    return {week: {pid: rec["pts"] for pid, rec in wk.items()} for week, wk in stats.items()}

def build_usage_to_date(year, max_weeks=MAX_WEEKS):
    """
    Returns: dict week -> dict player_id -> {feature: value} using only weeks 1..week-1.
    A game counts as played if Sleeper reports gp > 0 (or any usage stat is non-zero).
    Per-game values are divided by games played; snap_pct_to_date is the mean of the weekly
    off_snp / tm_off_snp over games played (weeks without team snap data are skipped).
    """
    stats = fetch_weekly_stats(year, max_weeks)
    tot = defaultdict(lambda: defaultdict(float))
    out = {}
    for week in range(1, max_weeks + 1):
        snap = {}
        for pid, t in tot.items():
            g = t["games"]
            if g <= 0:
                continue
            snap[pid] = {
                "rush_att_per_game_to_date": t["rush_att"] / g,
                "tgt_per_game_to_date": t["rec_tgt"] / g,
                "snap_pct_to_date": (t["snap_pct_sum"] / t["snap_games"]) if t["snap_games"] > 0 else np.nan,
                "kr_per_game_to_date": t["kr"] / g,
                "kr_yd_per_game_to_date": t["kr_yd"] / g,
                "pr_per_game_to_date": t["pr"] / g,
                "pr_yd_per_game_to_date": t["pr_yd"] / g,
            }
        out[week] = snap
        for pid, rec in stats.get(week, {}).items():
            played = rec["gp"] > 0 or any(rec[k] > 0 for k in ("rush_att", "rec_tgt", "off_snp", "kr", "pr"))
            if not played:
                continue
            t = tot[pid]
            t["games"] += 1
            for k in ("rush_att", "rec_tgt", "kr", "kr_yd", "pr", "pr_yd"):
                t[k] += rec[k]
            if rec["has_team_snaps"]:
                t["snap_pct_sum"] += rec["off_snp"] / rec["tm_off_snp"]
                t["snap_games"] += 1
    return out

# ====== Build dataset rows for starters from matchups ======
def build_dataset_for_league(league_id, year, adp_norm_map, players_meta, max_weeks=MAX_WEEKS):
    """
    Returns DataFrame with one row per player per week (both starters and bench).
    Adds a 'is_starter' column = 1 if starter that week, else 0.
    """
    weekly_points = build_weekly_player_points(year, max_weeks=max_weeks)
    usage_to_date = build_usage_to_date(year, max_weeks=max_weeks)
    rows = []
    
    for week in range(1, max_weeks + 1):
        try:
            matchups = requests.get(
                f"https://api.sleeper.app/v1/league/{league_id}/matchups/{week}", timeout=20
            ).json() or []
        except Exception:
            matchups = []

        for m in matchups:
            roster_id = m.get("roster_id")
            starters = set(str(pid) for pid in (m.get("starters") or []))
            all_players = set(str(pid) for pid in (m.get("players") or []))

            # Combine starters and bench; mark starters explicitly
            all_roster_players = all_players.union(starters)
            if not all_roster_players:
                continue

            for pid in all_roster_players:
                is_starter = 1 if pid in starters else 0
                usage = usage_to_date.get(week, {}).get(pid, {})

                # get points for this player this week
                pts_this_week = weekly_points.get(week, {}).get(pid, 0.0)
                pts_prev_1 = weekly_points.get(week - 1, {}).get(pid, 0.0) if week - 1 >= 1 else 0.0
                prev_weeks = [w for w in range(max(1, week - 3), week)]
                prev_points = [weekly_points.get(w, {}).get(pid, 0.0) for w in prev_weeks]
                pts_prev_3 = float(np.mean(prev_points)) if prev_points else 0.0

                played_sum = 0.0
                played_count = 0
                for w in range(1, week):
                    val = weekly_points.get(w, {}).get(pid)
                    if val is not None:
                        played_sum += val
                        played_count += 1
                pts_per_game_to_date = (played_sum / played_count) if played_count > 0 else 0.0

                meta = players_meta.get(pid, {}) or {}
                position = meta.get("position") or meta.get("pos") or meta.get("default_position") or "UNK"
                full_name = (meta.get("full_name") or f"{meta.get('first_name', '')} {meta.get('last_name', '')}").strip()
                pname_norm = normalize_name_for_match(full_name)
                adp_val = adp_norm_map.get(pname_norm, np.nan)

                # fallback for missing/invalid ADP
                if isinstance(adp_val, (str, list, dict)) or not isinstance(adp_val, (int, float)):
                    adp_val = 200.0
                if adp_val < 1:
                    adp_val = 200.0

                rows.append({
                    "season": year,
                    "week": week,
                    "roster_id": roster_id,
                    "player_id": pid,
                    "player_name": full_name,
                    "position": position,
                    "is_starter": is_starter,
                    "adp": float(adp_val),
                    "pts_prev_1": float(pts_prev_1),
                    "pts_prev_3": float(pts_prev_3),
                    "pts_per_game_to_date": float(pts_per_game_to_date),
                    **{f: usage.get(f, np.nan) for f in USAGE_FEATURES},
                    "points_this_week": float(pts_this_week)
                })
    return pd.DataFrame(rows)

# ====== MAIN: assemble dataset across seasons ======
players_meta = PLAYERS_META
all_rows = []

for year in YEARS:
    print("Processing season", year)
    # pick a league for user for that year (like your original code)
    league_id = None
    try:
        url = f"https://api.sleeper.app/v1/user/{USERNAME}/leagues/nfl/{year}"
        r = requests.get(url, timeout=20)
        if r.status_code == 200:
            leagues = r.json() or []
            if leagues:
                if LEAGUE_NAME_FILTER:
                    for L in leagues:
                        if LEAGUE_NAME_FILTER.lower() in (L.get("name") or "").lower():
                            league_id = L["league_id"]
                            break
                if not league_id:
                    league_id = leagues[0]["league_id"]
    except Exception:
        pass

    if not league_id:
        print("  no league found for", year, "- skipping")
        continue

    # fetch ADP for that year (use FFC for older, FantasyPros for future)
    if year >= 2021:
        adp_map = fetch_ffcalculator_adp(year, teams=FFC_TEAMS, scoring_format=FFC_SCORING)
    else:
        adp_map = fetch_fantasypros_adp(year, scoring=FFC_SCORING, position_scope="overall")
    # normalize keys
    adp_norm_map = {normalize_name_for_match(k): v for k, v in adp_map.items()}

    df_year = build_dataset_for_league(league_id, year, adp_norm_map, players_meta, max_weeks=MAX_WEEKS)
    print(f"  rows for {year}: {len(df_year)}")
    all_rows.append(df_year)
    time.sleep(0.2)

df = pd.concat(all_rows, ignore_index=True)
print("Total rows:", len(df))


In [ ]:
df = pd.concat(all_rows, ignore_index=True)
print("Total rows:", len(df))
print(df)
YEARS = [2021, 2022, 2023, 2024, 2025]
# Basic cleaning
df = df[df['points_this_week']!=0]
df = df[df["position"].notna() & (df["position"].str.upper() != "UNK")]
df["adp"] = df["adp"].fillna(200.0)
df["points_this_week"] = df["points_this_week"].astype(float)
df["pts_per_game_to_date"] = df["pts_per_game_to_date"].fillna(0.0)

# One-hot positions
pos_dummies = pd.get_dummies(df["position"].fillna("UNK"), prefix="pos")
df = pd.concat([df, pos_dummies], axis=1)

# Features and target
# Put the monotonic features early so it's easy to build the monotone vector
# Usage features are monotone-constrained (see below); NaN (no games yet) is handled natively by XGBoost
base_features = ["week", "adp", "pts_per_game_to_date"] + USAGE_FEATURES
X_base = df[base_features].copy()
X_other = df[pos_dummies.columns].copy()
X = pd.concat([X_base, X_other], axis=1)
y = df["points_this_week"].values

# Build monotone vector: we want
# - ADP: as ADP increases => predicted points should decrease  => monotone = -1
# - pts_per_game_to_date and every usage feature (snap %, carries, targets, kick/punt returns
#   and return yards per game): more => predicted points increase => monotone = +1
# All other features = 0
mono_vector = []
for f in X.columns:
    if f == "adp":
        mono_vector.append(-1)
    elif f in MONOTONE_INCREASING:
        mono_vector.append(+1)
    else:
        mono_vector.append(0)
mono_vector = tuple(mono_vector)
print("Monotone vector (len={}):".format(len(mono_vector)), mono_vector)

# Train / test by season: train on all seasons except the last, test on the last
train_seasons = sorted(df["season"].unique())[:-2]
test_season = sorted(df["season"].unique())[-2]
print("Training seasons:", train_seasons, "Test season:", test_season)

train_mask = df["season"].isin(train_seasons)
test_mask = df["season"] == test_season

X_train = X.loc[train_mask]
y_train = y[train_mask.values]
X_test = X.loc[test_mask]
y_test = y[test_mask.values]

print("Train rows:", X_train.shape[0], "Test rows:", X_test.shape[0])

# XGBoost regressor with monotonic constraints
model_params = dict(
    max_depth=5,
    learning_rate=0.1,
    objective="reg:squarederror",
    monotone_constraints=mono_vector,  # sklearn API accepts tuple/list
    tree_method="hist",
    random_state=42,
    verbosity=1
)

# Step 1: validation fit. Train on earlier seasons and early-stop on the held-out season
# to find the right number of trees and an honest out-of-sample RMSE.
print("Fitting validation model...")
val_model = xgb.XGBRegressor(n_estimators=1000, early_stopping_rounds=20, **model_params)
val_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=True)
best_n_estimators = val_model.best_iteration + 1

y_pred = val_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE (season {test_season}, out-of-sample): {rmse:.4f}  [best n_estimators = {best_n_estimators}]")

# Also report the seasons the validation model never saw
for s_ in sorted(df["season"].unique()):
    if s_ in train_seasons or s_ == test_season:
        continue
    m_ = (df["season"] == s_).values
    print(f"  Out-of-sample RMSE, season {s_}: {np.sqrt(mean_squared_error(y[m_], val_model.predict(X.loc[m_]))):.4f}")

# Step 2: final model = same settings, refit on ALL seasons (no holdout left, so no early stopping)
print(f"Refitting final model on all seasons {sorted(df['season'].unique())} ({len(X)} rows)...")
model = xgb.XGBRegressor(n_estimators=best_n_estimators, **model_params)
model.fit(X, y)

# Quick plot: predicted vs actual
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.4, s=12)
mx = max(y_test.max(), y_pred.max())
plt.plot([0, mx], [0, mx], 'r--')
plt.xlabel("Actual points")
plt.ylabel("Predicted points")
plt.title(f"Predicted vs Actual Points (RMSE={rmse:.2f})")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# Feature importance (gain)
try:
    xgb.plot_importance(model, importance_type="gain", max_num_features=12)
    plt.tight_layout()
    plt.show()
except Exception:
    pass

# Example predictions: show a few rows with features and predicted values
example_idx = df[test_mask].index[:8]
example_X = X.loc[example_idx]
preds = model.predict(example_X.values)
out_df = pd.concat([
    df.loc[example_idx, ["season","week","player_name","player_id","position","adp","pts_per_game_to_date","pts_prev_1","pts_prev_3"]].reset_index(drop=True),
    pd.DataFrame({"pred_points": preds})
], axis=1)
print("\nExample predictions (test season samples):")
print(out_df.head(8))

# Save model if desired
model.save_model("player_points_xgb.json")
print("Model saved to player_points_xgb.json")


In [ ]:
def get_roster_owner_map(league_id):
    """
    Returns a dict mapping roster_id -> owner display name (or username)
    for a given Sleeper league.
    """
    try:
        rosters = requests.get(f"https://api.sleeper.app/v1/league/{league_id}/rosters", timeout=20).json()
        users = requests.get(f"https://api.sleeper.app/v1/league/{league_id}/users", timeout=20).json()
    except Exception as e:
        print(f"Error fetching league data for {league_id}: {e}")
        return {}

    # Map owner_id -> display_name (fallback to username if needed)
    uid_to_name = {
        u["user_id"]: u.get("display_name") or u.get("username") or f"user_{u['user_id']}"
        for u in users
    }

    # Map roster_id -> owner display name
    rid_to_uid = {r["roster_id"]: r.get("owner_id") for r in rosters if r.get("owner_id")}
    roster_owner_map = {rid: uid_to_name.get(uid, f"user_{uid}") for rid, uid in rid_to_uid.items()}

    return roster_owner_map


In [ ]:
# ===== Week projection: player points, elimination / first-place odds, what-if lineups =====
# Requires the pipeline cell (helpers, PLAYERS_META, ADP) and the model cell (model, rmse) to have run.

# Sleeper designations that mean the player will not play -> projected 0.
# ("Questionable" / "Doubtful" are NOT adjusted: they are shown in the injury_status column instead.)
NOT_PLAYING = {"Out", "IR", "PUP", "Sus", "NA"}
FANTASY_POSITIONS = {"QB", "RB", "WR", "TE", "K", "DEF"}
SLOT_ELIGIBILITY = {"FLEX": {"RB", "WR", "TE"}, "SUPER_FLEX": {"QB", "RB", "WR", "TE"},
                    "REC_FLEX": {"WR", "TE"}, "WRRB_FLEX": {"WR", "RB"}}
PLAYER_COLS = ["season", "week", "roster_id", "owner_username", "player_id", "player_name", "position", "is_starter",
               "adp", "pts_per_game_to_date"] + USAGE_FEATURES

def save_csv(df, path):
    """Save a CSV; if the file is open in another program (Excel, a preview tab), save a timestamped copy instead."""
    try:
        df.to_csv(path, index=False)
        return path
    except PermissionError:
        alt = path.replace(".csv", f"_{time.strftime('%H%M%S')}.csv")
        df.to_csv(alt, index=False)
        print(f"NOTE: {path} is open in another program, so this was saved as {alt} instead.")
        return alt

def get_current_nfl_week():
    st = requests.get("https://api.sleeper.app/v1/state/nfl", timeout=20).json()
    return int(st["season"]), int(st["week"])

def get_bye_teams(year, week):
    """NFL teams with no game in `week` (empty set if the schedule can't be read)."""
    try:
        games = requests.get(f"https://api.sleeper.com/schedule/nfl/regular/{year}", timeout=20).json()
        everyone = {g["home"] for g in games} | {g["away"] for g in games}
        playing = {t for g in games if g["week"] == week for t in (g["home"], g["away"])}
        return (everyone - playing) if playing else set()
    except Exception:
        return set()

def find_league_id(year):
    leagues = requests.get(f"https://api.sleeper.app/v1/user/{USERNAME}/leagues/nfl/{year}", timeout=20).json() or []
    for L in leagues:
        if LEAGUE_NAME_FILTER.lower() in (L.get("name") or "").lower():
            return L["league_id"]
    return leagues[0]["league_id"] if leagues else None

def player_display_name(pid):
    meta = PLAYERS_META.get(pid, {}) or {}
    return (meta.get("full_name") or f"{meta.get('first_name', '')} {meta.get('last_name', '')}").strip()

def match_owner(name, owner_map, what="team"):
    """Case-insensitive username -> roster_id, restricted to the teams in owner_map."""
    hits = [rid for rid, u in owner_map.items() if str(u).lower() == str(name).strip().lower()]
    if not hits:
        raise ValueError(f"No {what} with username '{name}'. Options: {sorted(owner_map.values(), key=str.lower)}")
    return hits[0]

def add_projection(df, week, bye, apply_availability):
    """Adds raw_projection, nfl_team, injury_status, availability_flag and projected_points."""
    df = df.copy()
    feat_cols = list(model.get_booster().feature_names)
    pos = pd.get_dummies(df["position"], prefix="pos")
    X = pd.concat([df[["week", "adp", "pts_per_game_to_date"] + USAGE_FEATURES].reset_index(drop=True),
                   pos.reset_index(drop=True)], axis=1).reindex(columns=feat_cols, fill_value=0).astype(float)
    df["raw_projection"] = model.predict(X)
    df["nfl_team"] = df["player_id"].map(lambda p: (PLAYERS_META.get(p, {}) or {}).get("team"))
    df["injury_status"] = df["player_id"].map(lambda p: (PLAYERS_META.get(p, {}) or {}).get("injury_status"))
    df["availability_flag"] = np.where(df["nfl_team"].isin(bye), "BYE",
                                np.where(df["injury_status"].isin(NOT_PLAYING), df["injury_status"], ""))
    play = (df["availability_flag"] == "") if apply_availability else pd.Series(True, index=df.index)
    df["projected_points"] = np.where(play, df["raw_projection"], 0.0)
    return df

def build_week_frame(year, week, apply_availability=True):
    """Every rostered player on a team still alive in `week`, with features (weeks < week only) and projections."""
    league_id = find_league_id(year)
    adp_map = fetch_ffcalculator_adp(year)
    df_w = build_dataset_for_league(league_id, year, adp_map, PLAYERS_META, max_weeks=week)
    df_w = df_w[(df_w["week"] == week) & (df_w["player_id"] != "0")
                & df_w["position"].notna() & (df_w["position"].str.upper() != "UNK")].copy()
    df_w["adp"] = df_w["adp"].fillna(200.0)        # same fallback as training (200 = no ADP)
    owner_map = get_roster_owner_map(league_id)
    df_w["owner_username"] = df_w["roster_id"].map(owner_map)
    bye = get_bye_teams(year, week)
    df_w = add_projection(df_w[PLAYER_COLS], week, bye, apply_availability)
    league = requests.get(f"https://api.sleeper.app/v1/league/{league_id}", timeout=20).json()
    slots = [s for s in league.get("roster_positions", []) if s not in ("BN", "IR", "TAXI")]
    alive = {rid: owner_map.get(rid) for rid in df_w["roster_id"].unique()}
    ctx = {"year": year, "week": week, "league_id": league_id, "adp_map": adp_map, "bye": bye, "owner_map": owner_map,
           "alive": alive, "slots": slots, "apply_availability": apply_availability}
    return df_w.reset_index(drop=True), ctx

# ---------- what-if lineup moves ----------
_PLAYER_INDEX = {}
def _player_index():
    """normalized name -> Sleeper ids of current fantasy-relevant players (skips retired namesakes)."""
    if not _PLAYER_INDEX:
        for pid, info in PLAYERS_META.items():
            if info.get("position") not in FANTASY_POSITIONS or not (info.get("active") or info.get("team")):
                continue
            nm = normalize_name_for_match(player_display_name(pid))
            if nm:
                _PLAYER_INDEX.setdefault(nm, []).append(pid)
    return _PLAYER_INDEX

def resolve_player(query, df_w):
    """Name or Sleeper id -> Sleeper id. Rostered players are matched first, then the wider player pool."""
    q = str(query).strip()
    if q in PLAYERS_META:
        return q
    nm = normalize_name_for_match(q)
    rostered = df_w.loc[df_w["player_name"].map(normalize_name_for_match) == nm, "player_id"].unique().tolist()
    cands = rostered or _player_index().get(nm, [])
    if not cands:
        raise ValueError(f"Can't find a player named '{query}'. Check the spelling or pass a Sleeper player id.")
    if len(cands) > 1:
        opts = [f"{p} ({(PLAYERS_META.get(p, {}) or {}).get('position')}, {(PLAYERS_META.get(p, {}) or {}).get('team')})" for p in cands]
        raise ValueError(f"'{query}' matches several players: {opts}. Pass the Sleeper player id instead.")
    return cands[0]

def build_free_agent_rows(pids, ctx, roster_id, owner):
    """Feature rows for players not on any live roster, built exactly like build_dataset_for_league builds them."""
    year, week = ctx["year"], ctx["week"]
    weekly_points = build_weekly_player_points(year, max_weeks=week)
    usage_to_date = build_usage_to_date(year, max_weeks=week)
    rows = []
    for pid in pids:
        vals = [weekly_points.get(w, {}).get(pid) for w in range(1, week)]
        vals = [v for v in vals if v is not None]
        meta = PLAYERS_META.get(pid, {}) or {}
        name = player_display_name(pid)
        adp_val = ctx["adp_map"].get(normalize_name_for_match(name), np.nan)
        usage = usage_to_date.get(week, {}).get(pid, {})
        rows.append({"season": year, "week": week, "roster_id": roster_id, "owner_username": owner, "player_id": pid,
                     "player_name": name, "position": meta.get("position"), "is_starter": 1,
                     "adp": 200.0 if pd.isna(adp_val) else float(adp_val),
                     "pts_per_game_to_date": float(np.mean(vals)) if vals else 0.0,
                     **{f: usage.get(f, np.nan) for f in USAGE_FEATURES}})
    return pd.DataFrame(rows, columns=PLAYER_COLS)

def lineup_is_legal(positions, slots):
    """Can these starters (list of positions) fill the league's starting slots one-to-one?"""
    if len(positions) != len(slots):
        return False
    def fits(pos, slot): return pos in SLOT_ELIGIBILITY.get(slot, {slot})
    def assign(i, used):
        if i == len(positions):
            return True
        return any(j not in used and fits(positions[i], slots[j]) and assign(i + 1, used | {j}) for j in range(len(slots)))
    return assign(0, frozenset())

def apply_swaps(df_w, ctx, my_rid, swaps):
    """Copy of df_w with the lineup of `my_rid` changed by [(out, in), ...]."""
    d = df_w.copy()
    owner = ctx["alive"][my_rid]
    for out_q, in_q in swaps:
        out_pid, in_pid = resolve_player(out_q, d), resolve_player(in_q, d)
        out_mask = (d["roster_id"] == my_rid) & (d["player_id"] == out_pid)
        if not out_mask.any() or int(d.loc[out_mask, "is_starter"].iloc[0]) != 1:
            raise ValueError(f"'{out_q}' is not in {owner}'s starting lineup, so it can't be swapped out.")
        if out_pid == in_pid:
            raise ValueError(f"Swap '{out_q}' for itself?")
        in_rows = d[d["player_id"] == in_pid]
        d.loc[out_mask, "is_starter"] = 0          # bench the outgoing starter first (before any rows are appended)
        if len(in_rows):
            r = in_rows.iloc[0]
            if r["roster_id"] != my_rid:
                raise ValueError(f"'{in_q}' is on {r['owner_username']}'s roster, not yours. "
                                 f"To acquire him, use a 'trade' scenario.")
            if int(r["is_starter"]) == 1:
                raise ValueError(f"'{in_q}' is already in your starting lineup.")
            d.loc[in_rows.index, "is_starter"] = 1
        else:
            pos = (PLAYERS_META.get(in_pid, {}) or {}).get("position")
            if pos not in FANTASY_POSITIONS:
                raise ValueError(f"'{in_q}' is not a QB/RB/WR/TE/K/DEF.")
            fa = add_projection(build_free_agent_rows([in_pid], ctx, my_rid, owner), ctx["week"], ctx["bye"], ctx["apply_availability"])
            d = pd.concat([d, fa], ignore_index=True)
    starters = d[(d["roster_id"] == my_rid) & (d["is_starter"] == 1)]
    if not lineup_is_legal(starters["position"].tolist(), ctx["slots"]):
        raise ValueError(f"Illegal lineup after swaps: starters are {sorted(starters['position'].tolist())} "
                         f"but the league's slots are {ctx['slots']}.")
    return d

def best_lineup(team_df, slots, keep_bonus=0.0):
    """
    Best legal starting lineup for one team: the set of player_ids that fills every starting slot and maximises total
    projected points. keep_bonus > 0 makes current starters strongly preferred, so only the holes get re-filled.
    """
    from scipy.optimize import linear_sum_assignment
    players = team_df.reset_index(drop=True)
    NO = -1e6
    gain = np.full((len(players), len(slots)), NO)
    for i, r in players.iterrows():
        for j, s in enumerate(slots):
            if r["position"] in SLOT_ELIGIBILITY.get(s, {s}):
                gain[i, j] = r["projected_points"] + (keep_bonus if r["is_starter"] == 1 else 0.0)
    if len(players) < len(slots):
        raise ValueError(f"only {len(players)} players for {len(slots)} starting slots")
    rows, cols = linear_sum_assignment(-gain)
    if any(gain[i, j] <= NO / 2 for i, j in zip(rows, cols)):
        raise ValueError(f"no legal lineup: roster positions {sorted(players['position'].tolist())} cannot fill slots {slots}")
    return {players.loc[i, "player_id"] for i in rows}

def apply_trade(df_w, ctx, my_rid, trade):
    """
    Move players between MY_TEAM and the trade partner. The partner then plays its best legal lineup; my remaining
    starters stay in place and any hole is filled with my best legal option. Returns (frame, partner_roster_id).
    """
    d = df_w.copy()
    others = {r: u for r, u in ctx["alive"].items() if r != my_rid}
    partner = match_owner(trade["with"], others, "live team (other than yours)")
    partner_name, my_name = ctx["alive"][partner], ctx["alive"][my_rid]
    give = [resolve_player(q, d) for q in trade.get("give", [])]
    get = [resolve_player(q, d) for q in trade.get("get", [])]
    if not give and not get:
        raise ValueError("A trade needs at least one player in 'give' or 'get'.")
    if set(give) & set(get):
        raise ValueError("The same player can't be on both sides of the trade.")
    for pid in give:
        if not ((d["player_id"] == pid) & (d["roster_id"] == my_rid)).any():
            raise ValueError(f"'{player_display_name(pid)}' is not on {my_name}'s roster, so you can't trade him away.")
    for pid in get:
        if not ((d["player_id"] == pid) & (d["roster_id"] == partner)).any():
            raise ValueError(f"'{player_display_name(pid)}' is not on {partner_name}'s roster. "
                             f"(A free agent / waiver pickup is a swap, not a trade.)")
    for pid in give:
        d.loc[d["player_id"] == pid, ["roster_id", "owner_username", "is_starter"]] = [partner, partner_name, 0]
    for pid in get:
        d.loc[d["player_id"] == pid, ["roster_id", "owner_username", "is_starter"]] = [my_rid, my_name, 0]
    # the partner re-optimises fully; I keep my starters that are still here and only fill the holes
    for rid, bonus in ((partner, 0.0), (my_rid, 1e4)):
        mask = d["roster_id"] == rid
        try:
            chosen = best_lineup(d[mask], ctx["slots"], keep_bonus=bonus)
        except ValueError as e:
            raise ValueError(f"The trade leaves {ctx['alive'][rid]} without a legal lineup: {e}")
        d.loc[mask, "is_starter"] = d.loc[mask, "player_id"].isin(chosen).astype(int)
    return d, partner

def scenario_trades(spec):
    """All trades in a scenario, in order. 'trade' takes one trade (or a list); 'trades' takes a list."""
    trades = []
    for key in ("trade", "trades"):
        t = spec.get(key)
        if t:
            trades += list(t) if isinstance(t, (list, tuple)) else [t]
    return trades

def apply_scenario(df_w, ctx, my_rid, spec):
    """
    spec = a list of swaps, or {"trade": {...} and/or "trades": [{...}, {...}], "swaps": [...]}.
    Trades are applied in order, then the swaps. Returns (frame, [partner roster ids in trade order]).
    NOTE: a Python dict can't hold the same key twice (the last one silently wins), so use "trades": [...] for several.
    """
    if not isinstance(spec, dict):
        spec = {"swaps": spec}
    unknown = set(spec) - {"trade", "trades", "swaps"}
    if unknown:
        raise ValueError(f"Unknown scenario keys {sorted(unknown)}; use 'trade', 'trades' and/or 'swaps'.")
    d, partners = df_w, []
    for trade in scenario_trades(spec):
        d, partner = apply_trade(d, ctx, my_rid, trade)
        if partner not in partners:
            partners.append(partner)
    d = apply_swaps(d, ctx, my_rid, spec.get("swaps", []))
    return d, partners

def describe_scenario(spec):
    if not isinstance(spec, dict):
        spec = {"swaps": spec}
    parts = [f"TRADE with {tr['with']}: give {', '.join(tr.get('give', [])) or '-'} / get {', '.join(tr.get('get', [])) or '-'}"
             for tr in scenario_trades(spec)]
    if spec.get("swaps"):
        parts.append("; ".join(f"{o} -> {i}" for o, i in spec["swaps"]))
    return " | ".join(parts)

# ---------- simulation ----------
def simulate_week(df_w, k, immune_rids=(), n_sims=20000, seed=42):
    """
    Team score = sum of the starters' projected points; each available starter carries an independent error of about
    the model's out-of-sample RMSE. The same random draws are reused for every call with the same teams (same seed), so a
    what-if compared with the baseline differs only because of the lineup change, not because of simulation noise.
      prob_lowest     = P(lowest score of the week among all teams)
      prob_eliminated = P(cut): among teams that are NOT immune, the k lowest scores are eliminated
      prob_first      = P(highest score of the week among all teams, immune teams included)
    """
    st = df_w[df_w["is_starter"] == 1]
    teams = st.groupby("roster_id").agg(
        n_starters=("player_id", "size"),
        n_available=("projected_points", lambda s: int((s > 0).sum())),
        projected_points=("projected_points", "sum"),
    ).reset_index()
    model_rmse = float(globals().get("rmse", 7.5))
    teams["team_std"] = model_rmse * np.sqrt(teams["n_available"].clip(lower=1))
    T = len(teams)
    Z = np.random.default_rng(seed).standard_normal((T, n_sims))
    sims = teams["projected_points"].values[:, None] + teams["team_std"].values[:, None] * Z
    rank_all = sims.argsort(axis=0).argsort(axis=0)                 # 0 = lowest score in that simulation
    teams["prob_lowest"] = (rank_all == 0).mean(axis=1)
    teams["prob_first"] = (rank_all == T - 1).mean(axis=1)
    immune = teams["roster_id"].isin(list(immune_rids)).values
    k_eff = int(min(k, (~immune).sum()))
    rank_elig = np.where(immune[:, None], np.inf, sims).argsort(axis=0).argsort(axis=0)
    teams["prob_eliminated"] = ((rank_elig < k_eff) & ~immune[:, None]).mean(axis=1)
    teams["immune"] = immune
    teams["n_eliminated_assumed"] = k_eff
    return teams

def format_teams(teams, owner_map, year, week):
    teams = teams.copy()
    teams["owner_username"] = teams["roster_id"].map(owner_map)
    teams = teams.sort_values(["prob_eliminated", "prob_lowest"], ascending=False).reset_index(drop=True)
    teams.insert(0, "risk_rank", np.arange(1, len(teams) + 1))
    teams.insert(0, "week", week)
    teams.insert(0, "season", year)
    return teams[["season", "week", "risk_rank", "roster_id", "owner_username", "immune", "projected_points", "team_std",
                  "prob_eliminated", "prob_lowest", "prob_first", "n_eliminated_assumed", "n_starters", "n_available"]]

def cut_size(week, n_eliminated=None):
    return int(n_eliminated if n_eliminated is not None else ELIMINATIONS_BY_WEEK.get(week, 1))

def project_week(year, week, n_sims=20000, n_eliminated=None, immune_teams=(), apply_availability=True, seed=42):
    """Baseline projection: returns (players_df, teams_df, info). Uses only weeks before `week` for features."""
    df_w, ctx = build_week_frame(year, week, apply_availability)
    immune_rids = [match_owner(u, ctx["alive"], "live team") for u in immune_teams]
    k = cut_size(week, n_eliminated)
    teams = format_teams(simulate_week(df_w, k, immune_rids, n_sims, seed), ctx["owner_map"], year, week)
    players = df_w.sort_values(["roster_id", "is_starter", "projected_points"], ascending=[True, False, False]).reset_index(drop=True)
    players = players[["season", "week", "roster_id", "owner_username", "player_id", "player_name", "position", "nfl_team",
                       "is_starter", "injury_status", "availability_flag", "adp", "pts_per_game_to_date"] + USAGE_FEATURES
                      + ["raw_projection", "projected_points"]]
    info = {"league_id": ctx["league_id"], "bye_teams": sorted(ctx["bye"]), "n_eliminated": int(teams["n_eliminated_assumed"].iloc[0]),
            "n_active_teams": len(teams), "immune": [ctx["alive"][r] for r in immune_rids]}
    return players, teams, info


In [ ]:
# ===== Run the baseline projection =====
if TARGET_WEEK is None:
    _season, TARGET_WEEK = get_current_nfl_week()
    assert _season == TARGET_YEAR, f"Sleeper's current season is {_season}, not TARGET_YEAR={TARGET_YEAR}: set TARGET_WEEK explicitly"

frame, ctx = build_week_frame(TARGET_YEAR, TARGET_WEEK)
immune_rids = [match_owner(u, ctx["alive"], "live team") for u in IMMUNE_TEAMS]
K = cut_size(TARGET_WEEK, N_ELIMINATED)
teams_df = format_teams(simulate_week(frame, K, immune_rids, N_SIMS), ctx["owner_map"], TARGET_YEAR, TARGET_WEEK)
players_df = frame.sort_values(["roster_id", "is_starter", "projected_points"], ascending=[True, False, False])[
    ["season", "week", "roster_id", "owner_username", "player_id", "player_name", "position", "nfl_team", "is_starter",
     "injury_status", "availability_flag", "adp", "pts_per_game_to_date"] + USAGE_FEATURES + ["raw_projection", "projected_points"]]

print(f"{TARGET_YEAR} week {TARGET_WEEK}: {len(teams_df)} live teams, {K} eliminated this week | "
      f"immune: {[ctx['alive'][r] for r in immune_rids] or 'none'} | bye teams: {sorted(ctx['bye']) or 'none'}")

players_csv = f"week{TARGET_WEEK}_player_projections_{TARGET_YEAR}.csv"
teams_csv = f"week{TARGET_WEEK}_team_elimination_probabilities_{TARGET_YEAR}.csv"
players_csv = save_csv(players_df, players_csv)
teams_csv = save_csv(teams_df, teams_csv)
print(f"Saved {players_csv} ({len(players_df)} players) and {teams_csv} ({len(teams_df)} teams)")

show = teams_df.copy()
for c in ["prob_eliminated", "prob_lowest", "prob_first"]:
    show[c] = (show[c] * 100).round(1)
show[["projected_points", "team_std"]] = show[["projected_points", "team_std"]].round(1)
display(show[["risk_rank", "owner_username", "immune", "projected_points", "team_std", "prob_eliminated", "prob_lowest", "prob_first"]]
        .rename(columns={"prob_eliminated": "P(eliminated) %", "prob_lowest": "P(lowest score) %", "prob_first": "P(first place) %"}))


In [ ]:
# ===== What-if: change MY_TEAM's lineup and see the effect on elimination and first-place odds =====
my_rid = match_owner(MY_TEAM, ctx["alive"], "live team")
my_name = ctx["alive"][my_rid]

mine = frame[frame["roster_id"] == my_rid].sort_values(["is_starter", "projected_points"], ascending=[False, False])
print(f"{my_name}'s roster (starters first) - use these names in SCENARIOS:")
display(mine[["player_name", "position", "nfl_team", "is_starter", "injury_status", "availability_flag", "adp",
              "pts_per_game_to_date", "snap_pct_to_date", "projected_points"]].round(2).reset_index(drop=True))

def team_row(teams, rid):
    return teams.loc[teams["roster_id"] == rid].iloc[0]

base_teams = simulate_week(frame, K, immune_rids, N_SIMS)
base = team_row(base_teams, my_rid)
rows = [{"scenario": "Current lineup", "moves": "", "projected_points": base["projected_points"],
         "P(eliminated) %": base["prob_eliminated"] * 100, "P(lowest score) %": base["prob_lowest"] * 100,
         "P(first place) %": base["prob_first"] * 100}]
partner_rows = []
for name, spec in SCENARIOS.items():
    scen, partners = apply_scenario(frame, ctx, my_rid, spec)
    scen_teams = simulate_week(scen, K, immune_rids, N_SIMS)
    r = team_row(scen_teams, my_rid)
    row = {"scenario": name, "moves": describe_scenario(spec), "projected_points": r["projected_points"],
           "P(eliminated) %": r["prob_eliminated"] * 100, "P(lowest score) %": r["prob_lowest"] * 100,
           "P(first place) %": r["prob_first"] * 100}
    effects = []                       # a trade also changes the other team(s): show their without -> with trade
    for partner in partners:
        pb, pa = team_row(base_teams, partner), team_row(scen_teams, partner)
        effects.append(f"{ctx['alive'][partner]}: {pb['projected_points']:.1f} -> {pa['projected_points']:.1f} pts, "
                       f"P(elim) {pb['prob_eliminated'] * 100:.1f}% -> {pa['prob_eliminated'] * 100:.1f}%, "
                       f"P(last) {pb['prob_lowest'] * 100:.1f}% -> {pa['prob_lowest'] * 100:.1f}%, "
                       f"P(first) {pb['prob_first'] * 100:.1f}% -> {pa['prob_first'] * 100:.1f}%")
        for metric, col, scale in (("Projected points", "projected_points", 1), ("P(eliminated) %", "prob_eliminated", 100),
                                   ("P(last place) %", "prob_lowest", 100), ("P(first place) %", "prob_first", 100)):
            partner_rows.append({"scenario": name, "trade partner": ctx["alive"][partner], "metric": metric,
                                 "without trade": pb[col] * scale, "with trade": pa[col] * scale,
                                 "change": (pa[col] - pb[col]) * scale})
    row["trade partners (without -> with trade)"] = " | ".join(effects)
    rows.append(row)
whatif_df = pd.DataFrame(rows)
for c in ["P(eliminated) %", "P(lowest score) %", "P(first place) %"]:
    whatif_df["change in " + c.split(" %")[0]] = (whatif_df[c] - whatif_df[c].iloc[0]).round(2)
round_cols = ["projected_points", "P(eliminated) %", "P(lowest score) %", "P(first place) %"]
whatif_df[round_cols] = whatif_df[round_cols].round(2)
print(f"\n{my_name}: effect of lineup moves ({K} team(s) eliminated, immune: {[ctx['alive'][r] for r in immune_rids] or 'none'})")
if my_rid in immune_rids:
    print("NOTE: this team is immune, so its elimination probability is 0 in every scenario.")
display(whatif_df)
whatif_csv = save_csv(whatif_df, f"week{TARGET_WEEK}_scenario_comparison_{TARGET_YEAR}.csv")
print("Saved", whatif_csv)

# Trade partners: their chances of finishing last / first (and being cut) with and without each trade
partner_df = pd.DataFrame(partner_rows)
if len(partner_df):
    partner_df[["without trade", "with trade", "change"]] = partner_df[["without trade", "with trade", "change"]].round(2)
    print("\nTrade partners: without vs with the trade (same simulation draws, so the change is due to the trade)")
    display(partner_df)
    partner_csv = save_csv(partner_df, f"week{TARGET_WEEK}_trade_partner_effects_{TARGET_YEAR}.csv")
    print("Saved", partner_csv)


In [ ]:
# ===== Export this week's projections for the live view (live_app.py) =====
# Run this once before the games start (and again if rosters change a lot). The live app reads this file and combines it
# with Sleeper's live scores and the NFL game clocks, so it never has to run the model itself.
import datetime

def export_live_inputs(frame, ctx, year, week):
    # uncertainty per position = out-of-sample RMSE of starters (the validation model never saw these seasons)
    oos = df[~df["season"].isin(train_seasons) & (df["is_starter"] == 1)]
    err = pd.Series(oos["points_this_week"].values - val_model.predict(X.loc[oos.index]), index=oos.index)
    sigma = {pos: float(np.sqrt((e ** 2).mean())) for pos, e in err.groupby(oos["position"]) if len(e) >= 30}
    payload = {
        "season": year, "week": week, "league_id": ctx["league_id"],
        "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "scoring": "league scoring_settings applied to raw Sleeper stats (incl. return yards, INT -2, K and DEF rules)",
        "global_rmse": float(np.sqrt((err ** 2).mean())),
        "sigma_by_position": sigma,
        "eliminations_by_week": {str(w): int(n) for w, n in ELIMINATIONS_BY_WEEK.items()},
        "owners": {str(r): u for r, u in ctx["owner_map"].items()},
        "alive_roster_ids": [int(r) for r in ctx["alive"]],
        "position_mean_projection": frame[frame["is_starter"] == 1].groupby("position")["projected_points"].mean().round(3).to_dict(),
        "players": {
            p.player_id: {"name": p.player_name, "position": p.position, "team": p.nfl_team, "roster_id": int(p.roster_id),
                          "owner": p.owner_username, "projected_points": round(float(p.projected_points), 3),
                          "raw_projection": round(float(p.raw_projection), 3), "availability": p.availability_flag or "",
                          "injury_status": p.injury_status if isinstance(p.injury_status, str) else "",
                          "starter_at_export": int(p.is_starter)}
            for p in frame.itertuples()},
    }
    path = f"live_inputs_{year}_wk{week}.json"
    try:
        json.dump(payload, open(path, "w", encoding="utf-8"))
    except PermissionError:
        path = path.replace(".json", f"_{time.strftime('%H%M%S')}.json")
        json.dump(payload, open(path, "w", encoding="utf-8"))
        print("NOTE: the usual file is open in another program; saved as", path)
    return path, payload

live_path, live_payload = export_live_inputs(frame, ctx, TARGET_YEAR, TARGET_WEEK)
print(f"Saved {live_path}: {len(live_payload['players'])} players, sigma by position = "
      f"{ {k: round(v, 2) for k, v in live_payload['sigma_by_position'].items()} }")


In [ ]:
# ===== Backtest: how well do these projections match what actually happened? =====
# Uses project_week() on weeks that are already over (no availability adjustment: injury data is current-only).
def backtest_week(year, week):
    players, teams, info = project_week(year, week, apply_availability=False, n_sims=5000)
    m = requests.get(f"https://api.sleeper.app/v1/league/{info['league_id']}/matchups/{week}", timeout=20).json() or []
    actual = {x["roster_id"]: float(x.get("points") or 0) for x in m if x.get("players")}
    teams["actual_points"] = teams["roster_id"].map(actual)
    teams = teams.dropna(subset=["actual_points"])
    err = teams["actual_points"] - teams["projected_points"]
    lowest = teams.loc[teams["actual_points"].idxmin()]
    top = teams.loc[teams["actual_points"].idxmax()]
    return {"year": year, "week": week, "teams": len(teams),
            "corr(proj, actual)": round(float(np.corrcoef(teams["projected_points"], teams["actual_points"])[0, 1]), 2),
            "bias (actual-proj)": round(float(err.mean()), 1),
            "team RMSE": round(float(np.sqrt((err ** 2).mean())), 1),
            "std of z (want ~1)": round(float((err / teams["team_std"]).std()), 2),
            "actual lowest: risk rank": f"{int((teams['prob_lowest'] > lowest['prob_lowest']).sum()) + 1} of {len(teams)}",
            "actual top scorer: first-place rank": f"{int((teams['prob_first'] > top['prob_first']).sum()) + 1} of {len(teams)}"}

rows = [backtest_week(TARGET_YEAR, w) for w in range(2, TARGET_WEEK)]        # earlier weeks of this season (out-of-sample)
print("Weeks of the target season that are already over (the model never trained on these):")
display(pd.DataFrame(rows))
